In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

In [4]:
yellow_taxi_jan_2026 = pd.read_parquet(r"D:\projects\data analysis\NYC Taxi Data Analysis\data\raw_data\yellow_tripdata_2026-01.parquet")

In [5]:
yellow_taxi_jan_2026.shape

(3724889, 20)

# LOAD NYC TAXI ZONE SHAPEFILE


In [6]:
shapefile = r"D:\projects\data analysis\NYC Taxi Data Analysis\data\taxi_zones\taxi_zones.shp"
zones = gpd.read_file(shapefile)

print("\nTaxi zones loaded")
print("Number of zones:", len(zones))
print(zones.columns)


Taxi zones loaded
Number of zones: 263
Index(['OBJECTID', 'Shape_Leng', 'Shape_Area', 'zone', 'LocationID', 'borough',
       'geometry'],
      dtype='str')


In [7]:
zones.head()

,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough,geometry
0,1,0.116357,0.000782,Newark Airport,1,EWR,"POLYGON ((933100.918 192536.086, 933091.011 19..."
1,2,0.433470,0.004866,Jamaica Bay,2,Queens,"MULTIPOLYGON (((1033269.244 172126.008, 103343..."
2,3,0.084341,0.000314,Allerton/Pelham Gardens,3,Bronx,"POLYGON ((1026308.77 256767.698, 1026495.593 2..."
3,4,0.043567,0.000112,Alphabet City,4,Manhattan,"POLYGON ((992073.467 203714.076, 992068.667 20..."
4,5,0.092146,0.000498,Arden Heights,5,Staten Island,"POLYGON ((935843.31 144283.336, 936046.565 144..."


# zones = zones.to_crs(epsg=4326)

In [8]:
zones = zones.to_crs(epsg=4326)

In [9]:
zones.shape[0]

263

# CREATE REPRESENTATIVE POINT FOR EACH TAXI ZONE


In [10]:
# representative_point() guarantees that the point lies
# inside the polygon.


zones["representative_point"] = zones.geometry.representative_point()

zones["longitude"] = zones["representative_point"].x
zones["latitude"] = zones["representative_point"].y

In [11]:
zones.head(5)

,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough,geometry,representative_point,longitude,latitude
0,1,0.116357,0.000782,Newark Airport,1,EWR,"POLYGON ((-74.18445 40.695, -74.18449 40.6951,...",POINT (-74.17678 40.68952),-74.176778,40.689515
1,2,0.433470,0.004866,Jamaica Bay,2,Queens,"MULTIPOLYGON (((-73.82338 40.63899, -73.82277 ...",POINT (-73.82614 40.62572),-73.826141,40.625724
2,3,0.084341,0.000314,Allerton/Pelham Gardens,3,Bronx,"POLYGON ((-73.84793 40.87134, -73.84725 40.870...",POINT (-73.84948 40.86587),-73.849479,40.865871
3,4,0.043567,0.000112,Alphabet City,4,Manhattan,"POLYGON ((-73.97177 40.72582, -73.97179 40.725...",POINT (-73.97702 40.72415),-73.977024,40.724151
4,5,0.092146,0.000498,Arden Heights,5,Staten Island,"POLYGON ((-74.17422 40.56257, -74.17349 40.562...",POINT (-74.18994 40.55034),-74.189938,40.550339


In [12]:
drop_columns = ['OBJECTID', 'Shape_Leng', 'Shape_Area', 'LocationID',
       'geometry', 'representative_point']
yellow_taxi_jan_2026 = yellow_taxi_jan_2026.merge(zones, how='left', left_on='PULocationID', right_on='LocationID').drop(columns=drop_columns).rename(columns={
    "latitude": "pickup_latitude",
    "longitude": "pickup_longitude",
    "borough": "pickup_borough",
    "zone": "pickup_zone"
})
yellow_taxi_jan_2026 = yellow_taxi_jan_2026.merge(zones, how='left', left_on='DOLocationID', right_on='LocationID').drop(columns=drop_columns).rename(columns={
    "latitude": "dropoff_latitude",
    "longitude": "dropoff_longitude",
    "borough": "dropoff_borough",
    "zone": "dropoff_zone"
})

In [13]:
yellow_taxi_jan_2026.shape

(3724889, 28)

In [14]:
yellow_taxi_jan_2026.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,Airport_fee,cbd_congestion_fee,pickup_zone,pickup_borough,pickup_longitude,pickup_latitude,dropoff_zone,dropoff_borough,dropoff_longitude,dropoff_latitude
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,0.0,0.00,Upper West Side South,Manhattan,-73.978273,40.784107,Upper West Side North,Manhattan,-73.972814,40.791766
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,...,0.0,0.75,Midtown North,Manhattan,-73.978365,40.764424,Midtown East,Manhattan,-73.972145,40.756816
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,...,0.0,0.75,Central Park,Manhattan,-73.965572,40.782460,Upper East Side South,Manhattan,-73.965691,40.768542
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,0.0,0.75,Lincoln Square East,Manhattan,-73.981352,40.773906,Seaport,Manhattan,-74.002360,40.708490
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,...,0.0,0.75,Financial District South,Manhattan,-74.011308,40.703394,Little Italy/NoLiTa,Manhattan,-73.997407,40.720581


# checking which location is unknown or not present in zone file

In [15]:
yellow_taxi_jan_2026[['PULocationID','DOLocationID','pickup_zone','pickup_borough','pickup_longitude','pickup_latitude',
                      'dropoff_zone','dropoff_borough','dropoff_longitude','dropoff_latitude']].isna().sum()

PULocationID             0
DOLocationID             0
pickup_zone           5930
pickup_borough        5930
pickup_longitude      5930
pickup_latitude       5930
dropoff_zone         22302
dropoff_borough      22302
dropoff_longitude    22302
dropoff_latitude     22302
dtype: int64

In [16]:
zones.index

RangeIndex(start=0, stop=263, step=1)

In [17]:
print(yellow_taxi_jan_2026['PULocationID'].unique().min())
print(yellow_taxi_jan_2026['PULocationID'].unique().max())

1
265


# CALCULATE GEODESIC DISTANCE

In [22]:
import numpy as np

def calculate_great_circle_distance(df):
    lat1 = np.radians(df["pickup_latitude"].to_numpy())
    lon1 = np.radians(df["pickup_longitude"].to_numpy())

    lat2 = np.radians(df["dropoff_latitude"].to_numpy())
    lon2 = np.radians(df["dropoff_longitude"].to_numpy())

    R = 3958.7613  # Earth's radius in miles

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


In [23]:
yellow_taxi_jan_2026["great_circle_distance"] = calculate_great_circle_distance(
    yellow_taxi_jan_2026
)


In [24]:
yellow_taxi_jan_2026.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,pickup_zone,pickup_borough,pickup_longitude,pickup_latitude,dropoff_zone,dropoff_borough,dropoff_longitude,dropoff_latitude,geodesic_distance,great_circle_distance
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,Upper West Side South,Manhattan,-73.978273,40.784107,Upper West Side North,Manhattan,-73.972814,40.791766,0.601313,0.601313
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,...,Midtown North,Manhattan,-73.978365,40.764424,Midtown East,Manhattan,-73.972145,40.756816,0.618286,0.618286
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,...,Central Park,Manhattan,-73.965572,40.782460,Upper East Side South,Manhattan,-73.965691,40.768542,0.961648,0.961648
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,Lincoln Square East,Manhattan,-73.981352,40.773906,Seaport,Manhattan,-74.002360,40.708490,4.651695,4.651695
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,...,Financial District South,Manhattan,-74.011308,40.703394,Little Italy/NoLiTa,Manhattan,-73.997407,40.720581,1.392948,1.392948


In [30]:
yellow_taxi_jan_2026[['PULocationID','DOLocationID','pickup_zone','pickup_borough','pickup_longitude','pickup_latitude',
                      'dropoff_zone','dropoff_borough','dropoff_longitude','dropoff_latitude','great_circle_distance']].isna().sum()

PULocationID                 0
DOLocationID                 0
pickup_zone               5930
pickup_borough            5930
pickup_longitude          5930
pickup_latitude           5930
dropoff_zone             22302
dropoff_borough          22302
dropoff_longitude        22302
dropoff_latitude         22302
great_circle_distance    25251
dtype: int64